# Probability Distributions in Python
### Hypergeometric · Binomial · Poisson
Using `scipy.stats`, `numpy`, and `matplotlib`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import hypergeom, binom, poisson

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#f9f9f9',
    'axes.grid': True,
    'grid.alpha': 0.4,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'DejaVu Sans',
})

---
## 1. Hypergeometric Distribution

**Scenario:** Drawing cards from a deck *without* replacement.

**Parameters:**
- `M` — Population size (total cards: 52)
- `n` — Number of success states in population (e.g. 13 hearts)
- `N` — Number of draws (e.g. draw 5 cards)

**PMF:** $P(X=k) = \dfrac{\binom{n}{k}\binom{M-n}{N-k}}{\binom{M}{N}}$

**Mean:** $\mu = \dfrac{Nn}{M}$ &nbsp;&nbsp; **Variance:** $\sigma^2 = N\dfrac{n}{M}\dfrac{M-n}{M}\dfrac{M-N}{M-1}$

In [ ]:
# --- Parameters ---
M = 52   # population size (full deck)
n = 13   # success states (hearts)
N = 5    # draws

rv = hypergeom(M, n, N)
k = np.arange(0, N + 1)
pmf = rv.pmf(k)
cdf = rv.cdf(k)

print(f"Mean     : {rv.mean():.4f}")
print(f"Variance : {rv.var():.4f}")
print(f"Std Dev  : {rv.std():.4f}")
print(f"\nPMF values:")
for ki, pi in zip(k, pmf):
    print(f"  P(X={ki}) = {pi:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Hypergeometric Distribution  (M=52, n=13 hearts, N=5 draws)', fontsize=13, fontweight='bold', y=1.02)

# PMF
colors = ['#378ADD' if ki != k[np.argmax(pmf)] else '#185FA5' for ki in k]
axes[0].bar(k, pmf, color=colors, edgecolor='white', linewidth=0.8, width=0.6)
axes[0].set_title('PMF  —  P(X = k)', fontsize=11)
axes[0].set_xlabel('k (number of hearts drawn)')
axes[0].set_ylabel('Probability')
axes[0].set_xticks(k)
for ki, pi in zip(k, pmf):
    axes[0].text(ki, pi + 0.003, f'{pi:.3f}', ha='center', fontsize=9, color='#185FA5')

# CDF
axes[1].step(k, cdf, where='post', color='#378ADD', linewidth=2)
axes[1].fill_between(k, cdf, step='post', alpha=0.15, color='#378ADD')
axes[1].scatter(k, cdf, color='#185FA5', zorder=5, s=50)
axes[1].set_title('CDF  —  P(X ≤ k)', fontsize=11)
axes[1].set_xlabel('k')
axes[1].set_ylabel('Cumulative Probability')
axes[1].set_xticks(k)
axes[1].set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

In [ ]:
# --- Compare different draw sizes ---
fig, ax = plt.subplots(figsize=(11, 4.5))
draw_sizes = [3, 5, 8, 12]
palette = ['#B5D4F4', '#378ADD', '#185FA5', '#042C53']

for draws, color in zip(draw_sizes, palette):
    rv_i = hypergeom(M, n, draws)
    ki = np.arange(0, draws + 1)
    ax.plot(ki, rv_i.pmf(ki), 'o-', color=color, label=f'N={draws}', linewidth=1.8, markersize=6)

ax.set_title('Hypergeometric PMF — varying number of draws N', fontsize=12, fontweight='bold')
ax.set_xlabel('k (hearts drawn)')
ax.set_ylabel('P(X = k)')
ax.legend(title='Draws (N)')
plt.tight_layout()
plt.show()

---
## 2. Binomial Distribution

**Scenario:** Flipping a biased coin `n` times, counting heads.

**Parameters:**
- `n` — number of trials
- `p` — probability of success on each trial

**PMF:** $P(X=k) = \binom{n}{k} p^k (1-p)^{n-k}$

**Mean:** $\mu = np$ &nbsp;&nbsp; **Variance:** $\sigma^2 = np(1-p)$

In [ ]:
# --- Parameters ---
n = 20
p = 0.4

rv = binom(n, p)
k = np.arange(0, n + 1)
pmf = rv.pmf(k)
cdf = rv.cdf(k)

print(f"Mean     : {rv.mean():.4f}")
print(f"Variance : {rv.var():.4f}")
print(f"Std Dev  : {rv.std():.4f}")
print(f"Skewness : {(1 - 2*p) / rv.std():.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle(f'Binomial Distribution  (n={n}, p={p})', fontsize=13, fontweight='bold', y=1.02)

# PMF
mean_val = rv.mean()
bar_colors = ['#1D9E75' if abs(ki - mean_val) < 1 else '#9FE1CB' for ki in k]
axes[0].bar(k, pmf, color=bar_colors, edgecolor='white', linewidth=0.6, width=0.7)
axes[0].axvline(mean_val, color='#085041', linestyle='--', linewidth=1.5, label=f'Mean = {mean_val:.1f}')
axes[0].set_title('PMF  —  P(X = k)', fontsize=11)
axes[0].set_xlabel('k (number of successes)')
axes[0].set_ylabel('Probability')
axes[0].legend()

# CDF
axes[1].step(k, cdf, where='post', color='#1D9E75', linewidth=2)
axes[1].fill_between(k, cdf, step='post', alpha=0.15, color='#1D9E75')
axes[1].scatter(k, cdf, color='#085041', zorder=5, s=40)
axes[1].set_title('CDF  —  P(X ≤ k)', fontsize=11)
axes[1].set_xlabel('k')
axes[1].set_ylabel('Cumulative Probability')
axes[1].set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

In [ ]:
# --- Shape changes with p ---
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
probs = [0.2, 0.5, 0.8]
titles = ['Right-skewed  (p=0.2)', 'Symmetric  (p=0.5)', 'Left-skewed  (p=0.8)']
teal_shades = ['#9FE1CB', '#1D9E75', '#085041']

for ax, pi, title, color in zip(axes, probs, titles, teal_shades):
    rv_i = binom(n, pi)
    pmf_i = rv_i.pmf(k)
    ax.bar(k, pmf_i, color=color, edgecolor='white', linewidth=0.5, alpha=0.85, width=0.7)
    ax.axvline(rv_i.mean(), color='#04342C', linestyle='--', linewidth=1.4)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_xlabel('k')
    ax.set_ylabel('P(X = k)' if ax == axes[0] else '')

fig.suptitle('Binomial PMF — effect of p on shape  (n=20)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 3. Poisson Distribution

**Scenario:** Number of customer calls arriving per hour at a call center.

**Parameter:**
- `λ` (lambda) — average rate of events per interval

**PMF:** $P(X=k) = \dfrac{e^{-\lambda}\lambda^k}{k!}$

**Mean = Variance = λ** (unique property!)

In [ ]:
# --- Parameters ---
lam = 4.0

rv = poisson(lam)
k = np.arange(0, 20)
pmf = rv.pmf(k)
cdf = rv.cdf(k)

print(f"Mean     : {rv.mean():.4f}")
print(f"Variance : {rv.var():.4f}")
print(f"Std Dev  : {rv.std():.4f}")
print(f"Skewness : {1 / np.sqrt(lam):.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle(f'Poisson Distribution  (λ = {lam})', fontsize=13, fontweight='bold', y=1.02)

# PMF
bar_colors = ['#D85A30' if ki == int(lam) or ki == int(lam)+1 else '#F5C4B3' for ki in k]
axes[0].bar(k, pmf, color=bar_colors, edgecolor='white', linewidth=0.6, width=0.7)
axes[0].axvline(lam, color='#4A1B0C', linestyle='--', linewidth=1.5, label=f'λ = {lam}')
axes[0].set_title('PMF  —  P(X = k)', fontsize=11)
axes[0].set_xlabel('k (events per interval)')
axes[0].set_ylabel('Probability')
axes[0].legend()

# CDF
axes[1].step(k, cdf, where='post', color='#D85A30', linewidth=2)
axes[1].fill_between(k, cdf, step='post', alpha=0.15, color='#D85A30')
axes[1].scatter(k, cdf, color='#4A1B0C', zorder=5, s=40)
axes[1].set_title('CDF  —  P(X ≤ k)', fontsize=11)
axes[1].set_xlabel('k')
axes[1].set_ylabel('Cumulative Probability')
axes[1].set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

In [ ]:
# --- Poisson converges to Normal as λ grows ---
from scipy.stats import norm

lambdas = [1, 4, 10, 20]
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
coral_shades = ['#F5C4B3', '#F0997B', '#D85A30', '#993C1D']

for ax, lam_i, color in zip(axes, lambdas, coral_shades):
    k_i = np.arange(0, int(lam_i * 3) + 1)
    pmf_i = poisson(lam_i).pmf(k_i)
    ax.bar(k_i, pmf_i, color=color, edgecolor='white', linewidth=0.4, alpha=0.9, width=0.8)
    # overlay normal approximation
    x_cont = np.linspace(k_i[0], k_i[-1], 300)
    ax.plot(x_cont, norm.pdf(x_cont, lam_i, np.sqrt(lam_i)), color='#4A1B0C', linewidth=1.8, linestyle='--', label='Normal approx')
    ax.set_title(f'λ = {lam_i}', fontsize=11, fontweight='bold')
    ax.set_xlabel('k')
    if ax == axes[0]:
        ax.set_ylabel('P(X = k)')
        ax.legend(fontsize=8)

fig.suptitle('Poisson → Normal as λ increases', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4. Side-by-Side Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Hypergeometric
rv_h = hypergeom(52, 13, 10)
k_h = np.arange(0, 11)
axes[0].bar(k_h, rv_h.pmf(k_h), color='#378ADD', edgecolor='white', width=0.65)
axes[0].set_title('Hypergeometric\nM=52, n=13, N=10', fontsize=11, fontweight='bold')
axes[0].set_xlabel('k'); axes[0].set_ylabel('P(X = k)')

# Binomial
rv_b = binom(20, 0.4)
k_b = np.arange(0, 21)
axes[1].bar(k_b, rv_b.pmf(k_b), color='#1D9E75', edgecolor='white', width=0.65)
axes[1].set_title('Binomial\nn=20, p=0.4', fontsize=11, fontweight='bold')
axes[1].set_xlabel('k')

# Poisson
rv_p = poisson(4)
k_p = np.arange(0, 18)
axes[2].bar(k_p, rv_p.pmf(k_p), color='#D85A30', edgecolor='white', width=0.65)
axes[2].set_title('Poisson\nλ = 4', fontsize=11, fontweight='bold')
axes[2].set_xlabel('k')

fig.suptitle('PMF Comparison — Three Discrete Distributions', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Summary table
import pandas as pd
summary = pd.DataFrame({
    'Distribution': ['Hypergeometric', 'Binomial', 'Poisson'],
    'Mean':     [f'{rv_h.mean():.3f}', f'{rv_b.mean():.3f}', f'{rv_p.mean():.3f}'],
    'Variance': [f'{rv_h.var():.3f}',  f'{rv_b.var():.3f}',  f'{rv_p.var():.3f}'],
    'Std Dev':  [f'{rv_h.std():.3f}',  f'{rv_b.std():.3f}',  f'{rv_p.std():.3f}'],
    'Replacement': ['No', 'Yes', 'N/A (rare events)'],
})
summary